- 사용자 질문
        ↓
- [LLM] 어떤 도구를 써야 할지 판단
        ↓
- [Tool] 도구 실행 (검색, 계산 등)
        ↓
- [LLM] 결과를 보고 다음 행동 결정
        ↓
- 충분한 정보가 모이면 → 최종 답변

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent   # LangChain 1.x

In [2]:
# LLM 정의
llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
# 1. 도구 정의
@tool
def add(a: float, b: float) -> float:
    """두 숫자를 더합니다."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """두 숫자를 곱합니다."""
    return a * b

tools = [add, multiply]

In [4]:
# 2. 에이전트 생성
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="당신은 계산을 도와주는 어시스턴트입니다."
)

In [5]:
# 3. 실행
result = agent.invoke({
    "messages": [{"role": "user", "content": "3 더하기 5는 얼마야? 그리고 그 결과에 7을 곱하면?"}]
})

print(result["messages"][-1].content)

3 더하기 5는 8입니다. 그 결과인 8에 7을 곱하면 56이 됩니다.


In [6]:
result

{'messages': [HumanMessage(content='3 더하기 5는 얼마야? 그리고 그 결과에 7을 곱하면?', additional_kwargs={}, response_metadata={}, id='400d4f00-fe1f-4947-95ee-f14d22f1b0ec'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 105, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_17903ecff9', 'id': 'chatcmpl-DSAuMaNcr42UBK5YJhA5JOlfDpnIl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6a73-bac9-7630-9809-99da924f7909-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_vQQrPyXl4AaUqYxdQyvtNmko', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 8, 'b': 7}, 'id': 'call_g3xEudMYJn4D5

In [7]:
for msg in result["messages"]:
    print(f"[{msg.__class__.__name__}] {msg.content[:80] if msg.content else '(tool call)'}")

[HumanMessage] 3 더하기 5는 얼마야? 그리고 그 결과에 7을 곱하면?
[AIMessage] (tool call)
[ToolMessage] 8.0
[ToolMessage] 56.0
[AIMessage] 3 더하기 5는 8입니다. 그 결과인 8에 7을 곱하면 56이 됩니다.


In [8]:
# 가상의 result 데이터가 있다고 가정했을 때
messages = result["messages"]

for i, msg in enumerate(messages):
    role = msg.__class__.__name__  # 클래스 이름 (HumanMessage, AIMessage 등)
    content = msg.content
    
    # 1 & 2. AIMessage 처리 (content가 비어있고 tool_calls가 있는 경우)
    if role == "AIMessage":
        if not content and getattr(msg, 'tool_calls', None):
            display_content = "tool_calls"
        else:
            display_content = content
            
        # 4. 마지막 메시지가 AIMessage인 경우 추가 정보 출력
        if i == len(messages) - 1:
            model_provider = msg.response_metadata.get('model_provider', 'N/A')
            print(f"[{role}] {display_content} (Provider: {model_provider})")
            continue
            
        print(f"[{role}] {display_content}")

    # 3. ToolMessage 처리 (content + name)
    elif role == "ToolMessage":
        tool_name = getattr(msg, 'name', 'N/A')
        print(f"[{role}] {content} (Name: {tool_name})")
    
    # 기타 (HumanMessage 등)
    else:
        print(f"[{role}] {content}")


[HumanMessage] 3 더하기 5는 얼마야? 그리고 그 결과에 7을 곱하면?
[AIMessage] tool_calls
[ToolMessage] 8.0 (Name: add)
[ToolMessage] 56.0 (Name: multiply)
[AIMessage] 3 더하기 5는 8입니다. 그 결과인 8에 7을 곱하면 56이 됩니다. (Provider: openai)
